# Comandos Básicos de Pub Sub con Python

La biblioteca ***google-cloud-pubsub*** de Python es una interfaz para interactuar con Google Cloud Pub/Sub, un servicio de mensajería asíncrona que permite a los desarrolladores enviar y recibir mensajes entre aplicaciones independientes. Aquí te explico algunas de las funciones más utilizadas junto con ejemplos de cómo implementarlas:

In [ ]:
import os
from google.cloud import pubsub_v1
PROJECT_ID = os.getenv("GCP_PROJECT_ID")


: 

**1. Inicialización del Cliente**

Para interactuar con Google Cloud Pub/Sub, primero necesitas crear un cliente.

In [13]:
publisher_client = pubsub_v1.PublisherClient()
subscriber_client = pubsub_v1.SubscriberClient()
project_id = PROJECT_ID
my_topic = 'ssm-local-topic'
my_subs = 'sm-local-subscription'

**2. Crear un Tópico**

Un tópico es un canal a través del cual se envían los mensajes.

In [16]:
from google.api_core.exceptions import GoogleAPICallError

try:
    topic = publisher_client.create_topic(request={"name": topic_path})
    print(f"Created topic: {topic.name}")
except GoogleAPICallError as e:
    print(f"HTTP Status: {e.code}")
    print(f"Error Message: {e.message}")
    print(f"Details: {e.details}")
    print(f"\nFull Exception:\n{e}")

HTTP Status: 403
Error Message: User not authorized to perform this action.
Details: [reason: "IAM_PERMISSION_DENIED"
domain: "iam.googleapis.com"
metadata {
  key: "troubleshooter_url"
  value: "https://console.cloud.google.com/iam-admin/troubleshooter/summary;errorId=CiQwMWExN2ZmOC04MmEwLTcwYTQtYjQ5Zi00OTY2OTNlYzU4YzUSInByb2plY3RzL2FwcGxpZWQtcHJpbWFjeS01MTI2MTgtaDc%3D"
}
metadata {
  key: "resource"
  value: "projects/applied-primacy-512618-h7"
}
metadata {
  key: "permission"
  value: "pubsub.topics.create"
}
metadata {
  key: "error_info_id"
  value: "CiQwMWExN2ZmOC04MmEwLTcwYTQtYjQ5Zi00OTY2OTNlYzU4YzUSInByb2plY3RzL2FwcGxpZWQtcHJpbWFjeS01MTI2MTgtaDc="
}
]

Full Exception:
403 User not authorized to perform this action. [reason: "IAM_PERMISSION_DENIED"
domain: "iam.googleapis.com"
metadata {
  key: "troubleshooter_url"
  value: "https://console.cloud.google.com/iam-admin/troubleshooter/summary;errorId=CiQwMWExN2ZmOC04MmEwLTcwYTQtYjQ5Zi00OTY2OTNlYzU4YzUSInByb2plY3RzL2FwcGxpZWQtcHJpbW

In [14]:
topic_path = publisher_client.topic_path(project_id, my_topic)
print(f"Topic path: {topic_path}")
topic = publisher_client.create_topic(request={"name": topic_path})

Topic path: projects/applied-primacy-512618-h7/topics/ssm-local-topic


PermissionDenied: 403 User not authorized to perform this action. [reason: "IAM_PERMISSION_DENIED"
domain: "iam.googleapis.com"
metadata {
  key: "troubleshooter_url"
  value: "https://console.cloud.google.com/iam-admin/troubleshooter/summary;errorId=CiQwMWExN2ZmNy04MjVkLTc5NzAtYmY1My1iNGY5ODQ0MjQ2MmMSInByb2plY3RzL2FwcGxpZWQtcHJpbWFjeS01MTI2MTgtaDc%3D"
}
metadata {
  key: "resource"
  value: "projects/applied-primacy-512618-h7"
}
metadata {
  key: "permission"
  value: "pubsub.topics.create"
}
metadata {
  key: "error_info_id"
  value: "CiQwMWExN2ZmNy04MjVkLTc5NzAtYmY1My1iNGY5ODQ0MjQ2MmMSInByb2plY3RzL2FwcGxpZWQtcHJpbWFjeS01MTI2MTgtaDc="
}
]

**3. Publicar Mensajes**

Puedes publicar mensajes en un tópico. Los mensajes deben estar codificados en bytes.

In [ ]:
data = 'Hello, World 19!'
data = data.encode('utf-8')
future = publisher_client.publish(topic_path, data)
print(future.result())

11839887446719794


**4. Suscribirse a un Tópico**

Para recibir mensajes, necesitas suscribirte a un tópico creando una suscripción.

In [ ]:
subscription_path = f'projects/{project_id}/subscriptions/{my_subs}'
subscription = subscriber_client.create_subscription(request={"name": subscription_path, "topic": topic_path})

**5. Recibir Mensajes**

Una vez suscrito, puedes comenzar a recibir mensajes. Los mensajes son recibidos en un callback.

In [ ]:
import time

def callback(message):
    # Decode the message data to string
    message_data = message.data.decode('utf-8')
    print(f'Received message: {message_data}')
    message.ack()

# Subscribe to the topic
streaming_pull_future = subscriber_client.subscribe(subscription_path, callback=callback)
print(f'Listening for messages on {subscription_path}...')

try:
    # Keep the main thread alive to listen for messages
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    # Stop the subscriber when a KeyboardInterrupt is received
    streaming_pull_future.cancel()
    streaming_pull_future.result()  # Block until the shutdown is complete

Listening for messages on projects/smart-data-pub-sub/subscriptions/my-subscription...
Received message: Hello, World 19!
Received message: Hola, como vas?
Received message: todo bien?
Received message: aprendiendo servicios de mensajeria en GCP?


**6. Eliminar un Tópico o Suscripción**

Cuando ya no los necesites, puedes eliminar tanto tópicos como suscripciones.

Eliminar una suscripción:

In [ ]:
request = pubsub_v1.types.DeleteSubscriptionRequest(subscription=subscription_path)
subscriber_client.delete_subscription(request=request)

Eliminar un tópico:

In [ ]:
publisher_client.delete_topic(request={"topic": topic_path})